# 09 · Every sign in a whole photo (v3b)

Every notebook so far scored a **window**: a 128×128 crop holding at most one sign. A real GTSDB road photo
is 1360×800 and can hold 0–6 signs from 17 to 129 px across. v3b runs the v3a.1 model (`peak_decoy`) on whole
photos, **with no retraining**, and measures how many signs it finds. Four things change:

* **Several signs per photo.** The model has to answer more than once.
* **Stacked signs.** Many signs sit right next to another one, often on the same pole. The window cutter
  skipped every window with a second sign in view, so these were never scored before.
* **No single right scale.** The model learned signs 16–48 px across, but a sign in a photo can be 129 px.
* **Many more chances for a false alarm.** One photo covers about 87 windows' worth of road.

How v3b answers, in three steps (each section below is one of them):

1. **Pyramid.** Run the model on the photo at full size, half size and quarter size. Shrinking brings big
   signs into the size range the model knows.
2. **Peaks.** At each scale, every local maximum of the heatmap is a candidate, and a class is read there.
3. **Suppression.** A sign often peaks at more than one scale. Keep the highest-scoring candidate and
   drop any other that lands too close to it, regardless of class.

New words:

* **found**: some kept detection's point lies inside the sign's box **and** has the sign's class.
  **located** is the same but ignores class.
* **false positive (FP)**: a kept detection that finds no sign. Unlabeled real signs count too, because
  GTSDB only labels its 43 classes.
* **FPPI**: false positives per photo, the usual way to count false alarms when there are many signs per
  image.
* **τ_W**: the threshold where FPPI on the 600 dev photos is 0.5. It is chosen from false-positive scores
  alone. **τ_syn** is the run's window τ from notebook 07, kept for comparison.

**You'll see:** one photo going through all three steps live, how τ_W is chosen, what the pyramid does for
big signs, stacked signs across three seeds, what the false positives really are, and the six graded
predictions (W1–W6).

**Needs:** the GTSDB photos, the three `peak_decoy` runs, and their stored `detect_dev.json` reports
(`scripts/detect_photos.py`). **Writes:** nothing, unless you turn on a flag.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_decoy"  # the model to run live (v3a.1 seed 0); any run with a stored detect_dev.json works
SEED_RUNS = ("peak_decoy", "peak_decoy_s1", "peak_decoy_s2")  # the three seeds v3b graded, seed 0 first
PHOTO = 206  # the dev photo (0-599) to walk through: 5 signs from 23 to 110 px, found at all three scales
RUN_FOR_REAL = False  # True: re-score all 600 dev photos for RUN_NAME and re-grade W1-W5 (~4 min). REWRITES recorded results
SCORE_TEST = False  # True: score the 300 test photos ONCE for RUN_NAME (refused if already scored; all three seeds are)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import EVAL, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing; then load the model and the stored reports.
from findobject.data.gtsdb import DEV_PHOTOS, load_photo, read_gtsdb
from findobject.training.checkpoint import load_model
from findobject.utils.device import pick_device
from findobject.utils.files import sha256_file

GTSDB = ROOT / "data/raw/gtsdb/FullIJCNN2013"
require(GTSDB / "gt.txt", how="download GTSDB as described in README.md (Data section)")
if not 0 <= PHOTO < DEV_PHOTOS:
    raise ValueError(f"PHOTO must be a dev photo (0-{DEV_PHOTOS - 1}); test photos are only scored by the script, once")


def detect_cmd(run_dir, *extra):
    """The command that makes a run's stored whole-photo report, for require()'s message."""
    return f"python scripts/detect_photos.py --checkpoint {(run_dir / 'best.pt').relative_to(ROOT)} --split dev {' '.join(extra)}"


RUN_DIR = find_run(RUN_NAME)
SEED_DIRS = [find_run(name) for name in SEED_RUNS]
for d in [RUN_DIR, *SEED_DIRS]:
    require(EVAL / d.name / "detect_dev.json", how=detect_cmd(d))
require(EVAL / SEED_DIRS[0].name / "detect_dev_scale1.json", how=detect_cmd(SEED_DIRS[0], "--scales 1"))
require("results/v3b/grades_dev.json", "results/v3b/grades_test.json", "results/v3b/fp_audit.csv",
        "results/v3b/fp_sheet.png", how="these are committed with the repo; see RESULTS.md (v3b) for how they were made")

CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)
DEV = read_json(EVAL / RUN_DIR.name / "detect_dev.json")  # RUN_NAME's report on all 600 dev photos
if DEV["checkpoint_sha256"] != sha256_file(ROOT / CKPT):
    raise ValueError(f"{RUN_DIR.name}'s detect_dev.json was made with a different checkpoint; re-run: {detect_cmd(RUN_DIR)}")
SEED_DEV = [read_json(EVAL / d.name / "detect_dev.json") for d in SEED_DIRS]
TAU_SYN, TAU_W = DEV["thresholds"]["tau_syn"], DEV["thresholds"]["tau_w"]
photos = read_gtsdb(GTSDB)  # all 900 photos and their labeled signs; only dev photos are opened here

device = pick_device("auto")
model = load_model(CKPT, device)  # load_model returns the model already in eval mode
print(f"run {RUN_DIR.name}, τ_syn = {TAU_SYN:.3f}, τ_W = {TAU_W:.3f}, device {device}")
print(f"stored dev report: {DEV['report']['n_photos']} photos, {DEV['report']['n_signs']} signs, made on {DEV['device']}")

## 1. The pyramid: one photo at three sizes

The network is **fully convolutional** (notebook 02): it has no fixed input size, so it runs on the whole
1360×800 photo in one pass and gives a heatmap with one cell per 4×4 pixels (340×200 cells). The class head
is only read where the heatmap peaks, which is section 2.

The catch is size. Training signs were 16–48 px across. At scale 1 a 110 px sign is more than twice the
largest sign the model ever saw. Shrinking the photo to ½ or ¼ makes every sign 2 or 4 times smaller, so
each sign falls in the training range at *some* scale.

**Predict first:** for each sign in the table below, which scale do you think will find it?

In [ ]:
# The photo's labeled signs and how big each one is at every pyramid scale; * marks the training range (16-48 px).
from findobject.data.class_names import CLASS_NAMES
from findobject.eval.photo_truth import photo_truth
from findobject.inference.detection import PYRAMID_SCALES

TRAIN_MIN_PX, TRAIN_MAX_PX = 16, 48  # sign sizes in the training scenes (notebook 01)
photo = photos[PHOTO]
img = load_photo(photo)
truth = photo_truth(photo)  # the labels, exact repeats removed, each flagged stacked or not
print(f"photo {PHOTO}: {len(truth)} signs\n")
print(f"{'#':>2}  {'class':38} {'stacked':8}" + "".join(f"{'at ' + str(s):>9}" for s in PYRAMID_SCALES))
for s in truth:
    sizes = [s.long_side * scale for scale in PYRAMID_SCALES]
    marks = "".join(f"{v:7.0f}{'*' if TRAIN_MIN_PX <= v <= TRAIN_MAX_PX else ' '} " for v in sizes)
    print(f"{s.gt_index:>2}  {CLASS_NAMES[s.class_id][:38]:38} {str(s.stacked):8}{marks}")

In [ ]:
# Draw the photo with its labeled boxes, then the heatmap the model gives it at each scale.
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image


@torch.no_grad()
def heatmap_at(img, scale):
    """The heatmap's probabilities, [h, w], for the photo resized by `scale` (PIL bilinear, as detect() does)."""
    w, h = img.size
    small = img if scale == 1 else img.resize((round(w * scale), round(h * scale)), Image.BILINEAR)
    x = torch.from_numpy(np.asarray(small, dtype=np.uint8).copy()).permute(2, 0, 1).float().div(255)[None]
    return torch.sigmoid(model.heatmap_head(model.trunk(x.to(device))))[0, 0].float().cpu().numpy()


def draw_boxes(ax, signs, color="lime"):
    """Outline each labeled sign's box on a native-size photo."""
    for s in signs:
        x1, y1, x2, y2 = s.box
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=1.5))


W, H = img.size
fig, axes = plt.subplots(2, 2, figsize=(16, 9.6))
axes[0, 0].imshow(img)
draw_boxes(axes[0, 0], truth)
axes[0, 0].set_title(f"photo {PHOTO}, {W}×{H}, labeled signs in green")
for ax, scale in zip(axes.flat[1:], PYRAMID_SCALES):
    hm = heatmap_at(img, scale)
    ax.imshow(img)
    ax.imshow(hm, extent=(0, W, H, 0), cmap="magma", vmin=0, vmax=1, alpha=0.65)  # stretched back over the photo
    draw_boxes(ax, truth)
    ax.set_title(f"scale {scale}: model sees {round(W * scale)}×{round(H * scale)}, heatmap {hm.shape[1]}×{hm.shape[0]} cells, "
                 f"max {hm.max():.2f}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

Small signs glow at scale 1 and fade as the photo shrinks. Big signs do the opposite. Each scale is good at
a different band of sizes, and together they cover all of them.

## 2. Peaks, then suppression

`detect()` turns the heatmaps into an answer:

1. At each scale, a **peak** is a cell at least as high as its 8 neighbours, with probability ≥ 0.05.
   The class is read at the peak through the 3×3 class window, as in every version since v1.5.
2. Peaks from all three scales go into one list (**raw**).
3. **Suppression:** take the highest-scoring detection, drop every other one within 3 cells of it (in
   *its* scale's cells: 12 px at scale 1, 24 px at ½, 48 px at ¼), and repeat. Class is ignored. What's left
   is **kept**.

Then only kept detections with score ≥ τ_W count as the answer. Suppression runs *before* the threshold,
but that doesn't change anything above τ_W: a detection is only ever dropped by a higher-scoring one, and
anything higher than a detection above τ_W is above τ_W too.

In [ ]:
# Detect live: raw peaks at every scale, and what suppression keeps; then check against the stored dev run.
from collections import Counter

from findobject.inference.detect import detect

result = detect(model, img)  # .raw: every peak at every scale; .kept: after suppression, best first
raw_above = [d for d in result.raw if d.score >= TAU_W]
kept_above = [d for d in result.kept if d.score >= TAU_W]
print(f"raw peaks:  {len(result.raw):3} in all, {len(raw_above):2} at or above τ_W, by scale {dict(Counter(d.scale for d in raw_above))}")
print(f"kept:       {len(result.kept):3} in all, {len(kept_above):2} at or above τ_W, by scale {dict(Counter(d.scale for d in kept_above))}")

# The same photo in the stored dev report: the live detections above τ_W should be the same ones.
stored = next(p for p in DEV["photos"] if p["index"] == PHOTO)
stored_above = [k for k in stored["kept"] if k["score"] >= TAU_W]
same = len(stored_above) == len(kept_above) and all(
    (k["x"], k["y"], k["scale"], k["class_id"]) == (d.x, d.y, d.scale, d.class_id) and abs(k["score"] - d.score) < 1e-3
    for k, d in zip(stored_above, kept_above))
print(f"\nlive matches the stored dev report (made on {DEV['device']}): {same}")

In [ ]:
# Left: raw peaks above τ_W, coloured by scale. Right: what suppression keeps, with the square each one clears.
from findobject.inference.suppress import SUPPRESS_CELLS
from findobject.models.baseline import STRIDE

SCALE_COLOR = {1.0: "cyan", 0.5: "yellow", 0.25: "magenta"}
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
for ax, dets, title in ((axes[0], raw_above, "raw peaks"), (axes[1], kept_above, "kept after suppression")):
    ax.imshow(img)
    draw_boxes(ax, truth)
    for d in dets:
        ax.plot(d.x, d.y, "o", ms=7, mfc="none", mec=SCALE_COLOR[d.scale], mew=2)
        if dets is kept_above:
            r = SUPPRESS_CELLS * STRIDE / d.scale  # half-side of the square this detection clears
            ax.add_patch(plt.Rectangle((d.x - r, d.y - r), 2 * r, 2 * r, fill=False, ls="--", color=SCALE_COLOR[d.scale]))
    ax.set_title(f"{title} (score ≥ τ_W): {len(dets)}")
    ax.axis("off")
handles = [plt.Line2D([], [], ls="", marker="o", mfc="none", mec=c, mew=2, label=f"scale {s}") for s, c in SCALE_COLOR.items()]
axes[1].legend(handles=handles, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Matching: what counts as found, and what's a false positive

The answer is a list of **points** with a class each, not boxes. A detection **points at** a sign if its
point lies inside the sign's box. Matching is greedy in score order: the best detection claims the sign it
points at first, so a sign can only be found once.

Every kept detection that finds nothing is a false positive of one of three kinds:

* **duplicate**: points at a sign of its class that a better detection already found,
* **wrong_class**: points at a sign, but with a class none of the signs there have,
* **background**: points at no labeled sign. This includes real signs that GTSDB doesn't label.

In [ ]:
# Match the answer to the labels, then list every detection and every sign with what happened to it.
from findobject.eval.photo_match import match_photo

FP_COLOR = {"duplicate": "orange", "wrong_class": "orange", "background": "red"}


def answer_for(index):
    """(photo, labeled signs, detections above τ_W, their match) for one dev photo, computed live."""
    ph = load_photo(photos[index])
    signs = photo_truth(photos[index])
    dets = [d for d in detect(model, ph).kept if d.score >= TAU_W]
    return ph, signs, dets, match_photo(dets, signs)


def show_answer(ax, index):
    """Draw one dev photo's answer: green labeled boxes, cyan rings for finds, orange/red for false positives."""
    ph, signs, dets, m = answer_for(index)
    ax.imshow(ph)
    draw_boxes(ax, signs)
    for d, g, kind in zip(dets, m.found, m.fp_kind):
        color = "cyan" if g is not None else FP_COLOR[kind.value]
        ax.plot(d.x, d.y, "o", ms=8, mfc="none", mec=color, mew=2)
        ax.text(d.x + 10, d.y - 10, CLASS_NAMES[d.class_id][:22], color=color, fontsize=7,
                bbox=dict(fc="black", alpha=0.5, lw=0, pad=1))
    n_found = sum(g is not None for g in m.found)
    n_fp = len(dets) - n_found
    ax.set_title(f"photo {index}: {n_found} of {len(signs)} signs found, {n_fp} false positive{'' if n_fp == 1 else 's'}", fontsize=9)
    ax.axis("off")
    return signs, dets, m


_, _, dets, m = answer_for(PHOTO)
print("detections (best first):")
for d, g, loc, kind in zip(dets, m.found, m.located, m.fp_kind):
    what = f"found sign {g}" if g is not None else f"FALSE POSITIVE ({kind.value})"
    print(f"  ({d.x:6.0f}, {d.y:4.0f}) scale {d.scale:<4} score {d.score:.2f}  {CLASS_NAMES[d.class_id][:30]:30} -> {what}")
found_signs = {g for g in m.found if g is not None}
located_signs = {g for g in m.located if g is not None}
print("\nsigns:")
for s in truth:
    status = "found" if s.gt_index in found_signs else ("located, wrong class" if s.gt_index in located_signs else "MISSED")
    print(f"  {s.gt_index}: {s.long_side:3} px {'stacked ' if s.stacked else 'isolated'}  {CLASS_NAMES[s.class_id][:30]:30} {status}")

In [ ]:
# Draw the answer for PHOTO, plus photo 146, which goes wrong: its 125 px sign gets no detection at all,
# and one of the two small signs on the pole is located but given the wrong class.
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
show_answer(axes[0], PHOTO)
show_answer(axes[1], 146)
plt.tight_layout()
plt.show()

## 4. Choosing τ_W

Notebook 07 chose τ so that 1% of empty *windows* fire. A whole photo has many more places to fire, so v3b
chooses its own threshold on the 600 dev photos: **τ_W is the lowest threshold at which at most
0.5 × 600 = 300 false positives remain.**

Only false-positive scores go into that choice, never a sign's score. So τ_W can't be tuned to find more
signs: it only says how many false alarms we accept. Then τ_W is carried unchanged to the test photos.

The cell below recomputes τ_W from the stored false positives and checks it equals the stored one.

In [ ]:
# Recompute τ_W from the stored dev false positives alone, and check it is exactly the stored value.
from findobject.eval.froc import threshold_at_fppi

n_photos = len(DEV["photos"])
fp_scores = np.array([k["score"] for p in DEV["photos"] for k in p["kept"] if k["found"] is None], dtype=np.float32)
tau = threshold_at_fppi(fp_scores, n_photos, 0.5)
print(f"{len(fp_scores)} false positives on {n_photos} dev photos with score ≥ 0.05")
print(f"τ_W = {float(tau):.4f} leaves {(fp_scores >= tau).sum()} of them (budget 0.5 × {n_photos} = {int(0.5 * n_photos)})")
print(f"same as the stored τ_W: {float(tau) == TAU_W}")

In [ ]:
# The FROC curve: found (all 851 dev signs) against FPPI as the threshold falls, with the named thresholds marked.
froc, at = DEV["report"]["froc"], DEV["report"]["at"]
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(froc["fppi"], froc["recall"], lw=1.5)
for name, label in (("fppi_0.1", "FPPI 0.1"), ("tau_w", "τ_W (FPPI 0.5)"), ("tau_syn", "τ_syn"), ("fppi_1.0", "FPPI 1.0")):
    x, y = at[name]["fppi"]["value"], at[name]["groups"]["all"]["found"]["value"]
    ax.plot(x, y, "o", label=f"{label}: found {y:.1%} at FPPI {x:.2f}")
ax.legend(loc="lower right", fontsize=9)
ax.set(xlim=(0, 3), ylim=(0.7, 1.0), xlabel="false positives per photo (FPPI)", ylabel="found",
       title=f"{RUN_DIR.name}: FROC on dev; AP_point {DEV['report']['ap_point']:.3f}")
ax.grid(alpha=0.3)
plt.show()

The curve climbs steeply and then flattens. Going from 0.1 to 0.5 false positives per photo finds about
11 points more signs, but going from 0.5 to 1.0 adds less than half a point. τ_W sits near the knee.

## 5. What the pyramid buys (W3)

W3 predicted that the pyramid is what makes big signs work: for signs ≥ 64 px, at least 85% found with the
pyramid, and at most 60% at scale 1 alone. Below are seed 0's results both ways, by size, at τ_syn.

In [ ]:
# Found by size at τ_syn for seed 0: the full pyramid against scale 1 alone (detect_dev_scale1.json).
SCALE1 = read_json(EVAL / SEED_DIRS[0].name / "detect_dev_scale1.json")
pyr, one = SEED_DEV[0]["report"]["at"]["tau_syn"]["groups"], SCALE1["report"]["at"]["tau_syn"]["groups"]
print(f"{'group':22} {'n':>4}  {'pyramid':>8}  {'scale 1 only':>12}")
for group, label in (("small", "small (< 24 px)"), ("medium", "medium (24-47 px)"), ("large", "large (≥ 48 px)"),
                     ("ge64", "≥ 64 px (W3)"), ("all", "all")):
    a, b = pyr[group]["found"], one[group]["found"]
    print(f"{label:22} {a['n']:>4}  {a['value']:8.1%}  {b['value']:12.1%}")

In [ ]:
# Which scale found each sign, by sign size (RUN_NAME, kept detections at τ_W that found a sign).
from findobject.eval.real_report import size_bucket

by = Counter()
for p in DEV["photos"]:
    size = {s["gt_index"]: s["long_side"] for s in p["signs"]}
    for k in p["kept"]:
        if k["found"] is not None and k["score"] >= TAU_W:
            by[size_bucket(size[k["found"]]).value, k["scale"]] += 1
print(f"{'size':8}" + "".join(f"{'scale ' + str(s):>12}" for s in PYRAMID_SCALES))
for bucket in ("small", "medium", "large"):
    print(f"{bucket:8}" + "".join(f"{by[bucket, s]:>12}" for s in PYRAMID_SCALES))

Small signs are found at scale 1, medium signs mostly at scale 1 or ½, and large signs mostly at ½ or ¼.
At scale 1 alone, only 2 of the 139 signs ≥ 64 px are found. That's because the model has never seen a sign
that big, not because the sign is hard to see. W3 passed by a wide margin.

## 6. Stacked signs (W2)

A sign is **stacked** if another sign's center is closer than 1.5× its own size, as on a pole holding two
signs. W2 predicted that stacked signs would be found at least 5 points less often than isolated ones.
That gap would have meant training on scenes with several signs (v3c). It's measured in the medium bucket
only, because stacked signs tend to be smaller, and size alone would move the comparison.

**located before suppression** separates the two possible causes. If stacked signs are already missing
*before* suppression, the model itself is weak on them. If they're there before and gone after,
suppression deleted them.

In [ ]:
# Medium stacked vs isolated, for each seed at its τ_syn, with the photo-clustered 95% interval of the gap.
print(f"{'seed':5} {'isolated':>9} {'stacked':>8} {'gap':>7}   {'95% interval':16}  {'stacked located before / after supp.':>36}")
for i, d in enumerate(SEED_DEV):
    a = d["report"]["at"]["tau_syn"]
    g, diff = a["groups"], a["differences"]["medium_isolated_minus_stacked"]
    iso, st = g["medium_isolated"]["found"]["value"], g["medium_stacked"]["found"]["value"]
    before, after = g["stacked"]["located_before_suppression"]["value"], g["stacked"]["located"]["value"]
    print(f"{i:<5} {iso:9.1%} {st:8.1%} {100 * diff['value']:+6.1f}   [{100 * diff['ci'][0]:+5.1f}, {100 * diff['ci'][1]:+5.1f}]"
          f"   {before:>21.1%} / {after:.1%}")

In [ ]:
# Photo 73: six small signs, all of them stacked, close together on two poles.
fig, ax = plt.subplots(figsize=(12, 7.2))
show_answer(ax, 73)
plt.tight_layout()
plt.show()

For seed 0, stacked signs cost almost nothing (97.3% vs 97.6%), so W2 failed. The pre-registered decision
table reads this as "reading a class at every peak is enough: no multi-sign training needed". Suppression
isn't the problem either: located before and after suppression are nearly the same.

Seed 2 is the exception: a 9-point gap whose interval excludes 0, and the loss is already there before
suppression. So "no multi-sign training needed" holds for one training run, not all three. On test, seed 2
found 84 of 85 stacked signs, but test has too few stacked signs to settle it either way.

## 7. What the false positives really are

At τ_W each seed has 300 false positives on dev. The counts below say where they come from. Then come
the 40 highest-scoring seed-0 background false positives, cropped and labeled by hand without seeing the
score or photo (`results/v3b/fp_sheet.png`, labels in `fp_audit.csv`):

* **A**: a real sign outside the 43 classes (parking, crossing, motorway, ...)
* **B**: a real sign of one of the 43 classes that GTSDB forgot to label
* **C**: something sign-like that isn't a sign (traffic lights, shop boards, a number plate)
* **D**: nothing sign-like

In [ ]:
# False positives at τ_W by kind and by scale, for each seed; then the hand-audit labels for seed 0's top 40.
import csv

for i, d in enumerate(SEED_DEV):
    a = d["report"]["at"]["tau_w"]
    print(f"seed {i}: FPPI {a['fppi']['value']:.2f}, by kind {a['fp_by_kind']}, by scale {a['fp_by_scale']}")
with open(ROOT / "results/v3b/fp_audit.csv") as f:
    labels = Counter(row["label"] for row in csv.DictReader(f))
print("\nhand audit of the top 40:", {k: labels[k] for k in "ABCD"})
print(f"real signs (A + B): {labels['A'] + labels['B']} of {sum(labels.values())}")

In [ ]:
# The audit sheet itself: each tile is one false positive, numbered as in fp_audit.csv.
sheet = Image.open(ROOT / "results/v3b/fp_sheet.png")
fig, ax = plt.subplots(figsize=(14, 14 * sheet.height / sheet.width))
ax.imshow(sheet)
ax.axis("off")
plt.show()

More than half of the top false positives are real signs. The task still counts them as errors, because
the model named each one as one of the 43 classes. But firing on a parking sign is a different kind of
mistake from firing on a wall, so FPPI makes the model look somewhat worse than it is. Only 2 of 40 are
signs the dataset should have labeled.

## 8. The six predictions

All six were written down before any v3b code existed (`docs/v3b-spec.md` §2.6). W1–W5 were graded on dev.
The dev analysis was committed, and only then were the test photos scored, once per seed, for W6.

In [ ]:
# The stored grades: W1-W5 on dev, W6 on test.
dev_g, test_g = read_json("results/v3b/grades_dev.json")["grades"], read_json("results/v3b/grades_test.json")["grades"]
g = {**dev_g, **test_g}
lines = {
    "W1": f"whole photos vs windows, seed 0, same 645 signs: {g['W1']['whole_found']:.1%} vs {g['W1']['window_found']:.1%} (needed within 3 points)",
    "W2": f"medium isolated - stacked ≥ 5 points: seeds {', '.join(f'{100 * x:+.1f}' for x in g['W2']['gap_by_seed'])}",
    "W3": f"≥ 64 px: pyramid {g['W3']['pyramid']:.1%} (needed ≥ 85%), scale 1 alone {g['W3']['scale1']:.1%} (needed ≤ 60%)",
    "W4": f"seed 0 FPPI at τ_syn in [0.4, 1.6]: {g['W4']['fppi']:.3f}",
    "W5": f"3-seed mean found at τ_W ≥ 80%: {g['W5']['mean_found']:.1%}",
    "W6": f"test, 3-seed mean: FPPI {g['W6']['mean_test_fppi']:.3f} (needed ≤ 0.75), found {g['W6']['mean_test_found']:.1%} "
          f"(needed ≥ dev {g['W6']['mean_dev_found']:.1%} - 5 points)",
}
for w, text in lines.items():
    print(f"{w} {'PASS' if g[w]['pass'] else 'fail'}  {text}")
print(f"\n{sum(g[w]['pass'] for w in lines)} of {len(lines)} passed")

The two that failed were the two that expected trouble: a stacked-sign gap (W2), and false alarms at the
level the window results suggested (W4). Whole photos turned out easier than predicted on every count.

## Run it for real

These cells only print their commands unless you turn on a flag in the Settings cell.

* `RUN_FOR_REAL` re-scores all 600 dev photos for `RUN_NAME`, plus the scale-1-only run for seed 0's W3,
  and re-grades W1–W5 across `SEED_RUNS` (about 4 minutes in all). **This rewrites recorded results:**
  `outputs/eval/<run>/detect_dev*.json` and the committed `results/v3b/grades_dev.json`. That file records
  the SHA-256 of every report it graded, so even an identical re-run gets a new timestamp and a new hash.
  Use it for a new run, or revert the grade file afterwards with `git checkout results/v3b/grades_dev.json`.
* `SCORE_TEST` scores the 300 test photos once for `RUN_NAME`. All three seeds have already been scored, so
  for them the script refuses with exit code 2 and this cell stops with an error. That's the test guard
  doing its job.

In [ ]:
# Re-score dev for RUN_NAME (and scale 1 alone for seed 0), then re-grade W1-W5. Only runs when RUN_FOR_REAL = True.
run_script("detect_photos", "--checkpoint", CKPT, "--split", "dev", enabled=RUN_FOR_REAL)
if RUN_DIR == SEED_DIRS[0]:
    run_script("detect_photos", "--checkpoint", CKPT, "--split", "dev", "--scales", 1, enabled=RUN_FOR_REAL)
run_script("grade_v3b", "--runs", *[d.name for d in SEED_DIRS], "--split", "dev", enabled=RUN_FOR_REAL)

In [ ]:
# Score the whole test photos once for RUN_NAME. Only runs when SCORE_TEST = True.
run_script("detect_photos", "--checkpoint", CKPT, "--split", "test", enabled=SCORE_TEST)

That's v3b, and the end of the notebooks. `RESULTS.md` has the full v3b write-up, with every number and
interval. `docs/v3b-spec.md` is the plan as it was frozen before any code. `docs/wrap-up-ideas.md` collects
ideas for a demo and for finishing the project.